In [1]:
!pwd

/home/lab/rawhad/api-adapter


In [ ]:
# !uv add verifiers

Resolved 329 packages in 432ms                                       
Prepared 15 packages in 251ms                                            
Uninstalled 1 package in 0.30ms
Installed 21 packages in 6ms                                
 + aiofiles==25.1.0
 + aiolimiter==1.2.1
 + antlr4-python3-runtime==4.13.2
 ~ api-adapter==0.1.0 (from file:///workspace/home/lab/rawhad/api-adapter)
 + colorama==0.4.6
 + connect-python==0.9.0
 + griffe==1.15.0
 + latex2sympy2-extended==1.11.0
 + linkify-it-py==2.1.0
 + math-verify==0.9.0
 + mdit-py-plugins==0.5.0
 + openai-agents==0.10.5
 + prime-sandboxes==0.2.22
 + prime-tunnel==0.1.6
 + pyqwest==0.5.1
 + tenacity==9.1.4
 + textual==8.2.4
 + types-requests==2.33.0.20260408
 + uc-micro-py==2.0.0
 + verifiers==0.1.12
 + wget==3.2


In [3]:
import logging
from typing import Literal, cast

import verifiers as vf
from datasets import Dataset, load_dataset

logger = logging.getLogger("verifiers.ifbench")

In [179]:
# my dataset needs to have "question", and "answer" columns. "info" is optional.

In [4]:
# convert dataset to this format
dataset = Dataset.from_json('data/ifbench/input_train_data_with_claude_response_5000_subset.jsonl')
dataset

Dataset({
    features: ['key', 'messages', 'ground_truth', 'dataset', 'constraint_type', 'constraint', 'claude_response', 'claude_reward'],
    num_rows: 5000
})

In [5]:
dataset = dataset.map(lambda x: {
    "question": (
        f"User Prompt: {x['messages'][0]['content']}\n<draft_response>{x['claude_response']}</draft_response>\n"
        "/no_think"
    ),
    "answer": "",
    "info": {**x}  # answer is empty, info contains fields for verification
})
dataset


Dataset({
    features: ['key', 'messages', 'ground_truth', 'dataset', 'constraint_type', 'constraint', 'claude_response', 'claude_reward', 'question', 'answer', 'info'],
    num_rows: 5000
})

In [6]:
# split dataset into train and val
# 80-20 stratified split on claude_reward values
lgtm_dataset = dataset.filter(lambda x: x['claude_reward'] == True)
fixme_dataset = dataset.filter(lambda x: x['claude_reward'] == False)

lgtm_train_dataset, lgtm_val_dataset = lgtm_dataset.train_test_split(test_size=0.2, seed=42).values()
fixme_train_dataset, fixme_val_dataset = fixme_dataset.train_test_split(test_size=0.2, seed=42).values()

from datasets import concatenate_datasets
train_dataset = concatenate_datasets([lgtm_train_dataset, fixme_train_dataset])
val_dataset = concatenate_datasets([lgtm_val_dataset, fixme_val_dataset])

train_dataset, val_dataset

(Dataset({
     features: ['key', 'messages', 'ground_truth', 'dataset', 'constraint_type', 'constraint', 'claude_response', 'claude_reward', 'question', 'answer', 'info'],
     num_rows: 3999
 }),
 Dataset({
     features: ['key', 'messages', 'ground_truth', 'dataset', 'constraint_type', 'constraint', 'claude_response', 'claude_reward', 'question', 'answer', 'info'],
     num_rows: 1001
 }))

In [7]:
dataset[0]['question']

'User Prompt: Identificeer welk instrument een snaar- of slaginstrument is: Kpanlogo, Shamisen. All sentences must be connected using hyphens, with no spaces between them. The last word of your response should be the word brief.\n<draft_response>Shamisen-is-een-snaarinstrument-en-Kpanlogo-is-een-slaginstrument-De-Shamisen-stamt-uit-Japan-en-heeft-drie-snaren-De-Kpanlogo-is-een-Ghanese-hand-drum-Dit-antwoord-is-brief</draft_response>\n/no_think'

In [8]:
# gepa optimized system prompt
SYSTEM_PROMPT = """
You are a helpful assistant. Your job is to look at the user prompt and the draft response and determine if the draft response is correct.

You MUST think carefully inside your reasoning before outputting your final answer. Follow these evaluation steps:

**Step 1 - Identify All Constraints**: Read the user prompt thoroughly and list EVERY explicit constraint, formatting requirement, and instruction. Be exhaustive — but ONLY include constraints that are explicitly stated in the prompt. Do NOT invent or infer constraints that are not present. Common constraint types include:
- Required keywords that must appear (with specific frequencies) or must NOT appear
- Word count, sentence count, paragraph count, section count, or bullet point count requirements
- Structural formatting (titles wrapped in specific markers, sections with specific labels, bullet points, headers, bigram wrapping in double angular brackets, square brackets around words)
- Capitalization rules (e.g., all caps, capital word frequency minimums)
- Starting/ending word constraints for sentences or the overall response
- Language requirements
- Inclusion of specific elements (palindromes, postscripts, placeholders in square brackets)
- Punctuation rules (e.g., no exclamation marks, no dots, hyphens between sentences)
- Unique word constraints (no repeated words)
- Letter frequency constraints (e.g., letter X should appear fewer than N times)
- Copy/repeat instructions (e.g., "repeat the request without change and do not answer")
- JSON formatting requirements
- Paragraph separation requirements (e.g., two new lines between paragraphs)
- Adjacent word letter constraints
- Character index span copying
- Phrase repetition with transformation
- Nth paragraph first word requirements
- Any other explicit formatting or content instructions

**Step 2 - Check Content Correctness**: Verify that the draft response properly addresses the user's question or request with factually accurate information, correct mathematical calculations, and sound logical reasoning. A response that is just an error message, blank, or the word "Error" is NOT correct — it fails to address the actual request. Also check if the prompt instructs NOT to answer and only to repeat — in that case, answering the question is incorrect.

**Step 3 - Verify Each Constraint Individually**: Go through EVERY SINGLE constraint identified in Step 1 and explicitly check whether the draft response satisfies it. Be meticulous and skeptical:
- Count words, sentences, paragraphs exactly — do not estimate
- Count keyword appearances exactly — search the entire response carefully
- Count letter occurrences exactly when letter frequency constraints exist
- Verify structural elements character by character (bigram wrapping, square brackets, title markers)
- Check paragraph separators match requirements (e.g., markdown divider `***` or `\n\n`)
- Verify keyword positions (e.g., "keyword X as the Nth word of sentence M")
- Check start/end words of sentences and of the entire response
- Validate any JSON formatting
- For character index span copying, count characters in the original prompt carefully starting from index 0
- For keyword frequency constraints, count the EXACT number of times a keyword appears — not more, not less
- For "no two adjacent words start with consecutive letters" constraints, check EVERY pair of adjacent words
- For phrase repetition constraints, verify the exact number of repetitions AND that transformations follow the rules
- For paragraph first-word constraints, identify paragraphs correctly based on the specified separator and check the first word of the specified paragraph
Note each constraint as SATISFIED or VIOLATED with a brief explanation.

**Step 4 - Make Your Decision**:
- If the draft response is correct in content AND satisfies ALL constraints with zero violations, output exactly:
  <|ADAPTER_RESPONSE_START|>CORRECT<|ADAPTER_RESPONSE_END|>

- If the draft response has ANY content error OR ANY constraint violation, provide a corrected response that fixes ALL issues while preserving what was already correct:
  <|ADAPTER_RESPONSE_START|>[your corrected response here]<|ADAPTER_RESPONSE_END|>

**Critical Rules**:
- Tag formatting is paramount: use exactly <|ADAPTER_RESPONSE_START|> and <|ADAPTER_RESPONSE_END|> with the pipe characters and angle brackets precisely as shown. Double-check your tags character by character before outputting. The opening tag must be <|ADAPTER_RESPONSE_START|> and the closing tag must be <|ADAPTER_RESPONSE_END|>. Any typo (e.g., missing pipe character, swapped brackets like |< instead of <|, missing | before >) will cause a catastrophic failure.
- A draft response that is just "Error" or blank or fails to address the request is almost NEVER correct. Always provide a proper corrected response in such cases.
- Do NOT invent constraints that are not explicitly stated in the user prompt. Only check for constraints that are actually written in the prompt. For example, if the prompt only says "no dots," do not also add "no commas" or "no hyphens" as constraints.
- Do NOT say CORRECT if ANY constraint is violated, even a minor one. When in doubt, re-count and re-verify.
- Do NOT unnecessarily correct responses that are already correct. If the content is accurate and genuinely ALL constraints are met after careful verification, output CORRECT. Do not make changes just because you think something could be "better" — only fix actual violations.
- When providing a corrected response, ensure it satisfies ALL identified constraints from the user prompt simultaneously. Your corrected response replaces the draft entirely, so it must be complete and self-contained.
- If the draft appropriately refuses a harmful, dangerous, or unethical request, treat the refusal as correct behavior even if some formatting constraints from the malicious prompt are not followed.
- Pay special attention to constraints that are easy to overlook: keyword frequency/position requirements, exact paragraph counts, bigram wrapping, letter frequency limits, copy/repeat instructions, structural formatting details, nth paragraph first word requirements, and phrase repetition with transformation rules.
- When counting paragraphs, use the separator specified in the prompt (e.g., two new lines). If no separator is specified, use standard paragraph breaks. Be precise about which paragraph is which.
- Your corrected response should go directly inside the tags with no additional commentary outside them.
- Before finalizing, re-read your output to confirm the tags are exactly correct: <|ADAPTER_RESPONSE_START|> to open and <|ADAPTER_RESPONSE_END|> to close. Verify both tags character by character.
"""

In [9]:
# response parser
from loguru import logger as loguru_logger
import re
ptrn = re.compile(r"<\|ADAPTER_RESPONSE_START\|>(.*)<\|ADAPTER_RESPONSE_END\|>", re.DOTALL)
def extract_adapter_response(response):
    try: return ptrn.findall(response)[-1]
    except:
        loguru_logger.exception(f"Error in extract_adapter_response")
        return ""

parser = vf.MaybeThinkParser(extract_fn=extract_adapter_response)
parser


In [10]:
from loguru import logger as loguru_logger

from src.api_adapter.ifbench.eval_utils import (
    InputExample,
    test_instruction_following_loose,
    normalize_instruction_kwargs,
)


def reward_fn(
    completion: vf.Messages, parser: vf.Parser, state: vf.State, info: vf.Info, **kwargs
) -> float:
    """
    - if response is CORRECT, and claude_reward is True, return 1.0
    - if response is CORRECT, and claude_reward is False, return 0.0
    - if response is an actual_answer,
        - and is same as draft_response, return 0.0  # we dont want the model to repeat the draft response
        - and is different from draft_response
            - and claude_reward is True, return 0.0  # because we want to punish the model for saying that draft response was incorrect.
            - and is incorrect, return 0.5  # we want to reward the model for saying that draft response was incorrect but also punish it slightly for generating the wrong answer.
            - and is correct, return 1.0  # we want to reward the model for saying that draft response was incorrect and generating the correct answer.

    - if claude_reward is True,
        - and response is CORRECT, return 1.0
        - and response is not CORRECT, return 0.0

    if response is CORRECT, and claude_reward is False, return 0.0
    if response is actual_answer,
        - and is same as draft_response, return 0.0
        - and is different from draft_response
            - and is incorrect, return 0.5
            - and is correct, return 1.0

    where,
    - response is the response from the model
    - claude_reward is the reward for the claude response
    - "CORRECT" is model's way of saying LGTM
    - "actual_answer" is the fixed final answer that the model generates
    """
    try:
        loguru_logger.debug(f'Completion: {completion[-1].content}')
        response = parser.parse_answer(completion) or ""
        loguru_logger.debug(f'Response: {response}')
        gt = eval(info['ground_truth'])
        loguru_logger.debug(f'GT: {gt}')
        input_example = InputExample(
            key=info["key"],
            instruction_id_list=gt[0]['instruction_id'],
            prompt=info['messages'][-1]['content'],
            kwargs=normalize_instruction_kwargs(gt[0]['kwargs']),
        )
        loguru_logger.debug(f'claude reward: {info["claude_reward"]}')

        # logs to state
        state['lgtm_count'] = 1 if response.strip() == 'CORRECT' else 0
        state['fixme_count'] = 1 if response.strip() != 'CORRECT' else 0

        if info['claude_reward']:
            if response.strip() == "CORRECT": return 1.0
            else: return 0.0

        if response.strip() == "CORRECT": return 0.0
        if response.strip() == info['claude_response']: return 0.0

        reward = 0.5
        prompt_to_response = {input_example.prompt: response}
        output_example = test_instruction_following_loose(input_example, prompt_to_response)
        loguru_logger.debug(f'output_example: {output_example}')
        return reward + (float(output_example.follow_all_instructions) / 2)
    except:
        loguru_logger.exception(f"Error in reward_fn")
        logger.exception(f"Error in reward_fn")
        return 0.0

def get_lgtm_count(state: vf.State, **kwargs): return state['lgtm_count']
def get_fixme_count(state: vf.State, **kwargs): return state['fixme_count']



# test reward function
from verifiers.types import AssistantMessage

completions = [
    AssistantMessage(
        role="assistant",
        content="<think>\n\n</think>\n\n**Step 1 - Identify All Constraints**:\n- The response should have 9 paragraphs.\n- Paragraphs are separated with the markdown divider: ***.\n- The first word of the response should be the word \"maximum\".\n\n**Step 2 - Check Content Correctness**:\n- The content correctly calculates the number of tasks the team should aim to complete this month.\n- The calculation is accurate: 60% of 120 tasks = 72 tasks completed last month.\n- A 25% improvement on the completion rate (60% + 25% of 60% = 75%) results in 75% of 120 tasks = 90 tasks.\n- The reasoning is sound and logically follows from the given information.\n\n**Step 3 - Verify Each Constraint Individually**:\n- **Number of paragraphs**: The draft response has 9 paragraphs, which satisfies the constraint.\n- **Paragraph separator**: Each paragraph is separated by the markdown divider `***`, which satisfies the constraint.\n- **First word**: The first word of the response is \"Maximum\", which satisfies the constraint.\n\n**Step 4 - Make Your Decision**:\n- The draft response is correct in content and satisfies all constraints.\n\n<|ADAPTER_RESPONSE_START|>CORRECT<|ADAPTER_RESPONSE_END|>",
    )
]
info = {'claude_response': "Maximum effort is required to turn around a failing product, and understanding the numerical targets is essential for success. The project manager must calculate precisely how many tasks the team should aim to complete this month based on their previous performance and improvement goals. This mathematical problem requires careful analysis of both the historical completion rates and the projected growth targets.\n\n***\n\nLast month, the team's performance was disappointing, with only 60% task completion. Starting with 120 total tasks assigned for the month, this represents a significant shortfall that prompted the need for improvement strategies. The project manager recognized this failure rate and decided that action was necessary to get the team back on track.\n\n***\n\nTo determine the actual number of tasks completed last month, we must calculate 60% of the 120 assigned tasks. This calculation is straightforward: 0.60 multiplied by 120 equals 72 tasks. Therefore, the team successfully completed 72 tasks while failing to complete 48 tasks, leaving a notable gap in productivity.\n\n***\n\nThis month, the team has set an ambitious goal to improve their performance by 25%. This improvement target represents management's commitment to better execution and team efficiency. The question now becomes how to apply this 25% improvement to the existing completion rate of 60%.\n\n***\n\nWhen we say the team wants to improve their performance by 25%, this means they want to increase their completion rate by 25% of the current rate, not by adding 25 percentage points. In other words, the improvement is calculated as a percentage of the existing 60% completion rate. This is the standard interpretation of such improvement goals in business contexts.\n\n***\n\nTo calculate the new target completion rate, we take the previous month's completion rate of 60% and multiply it by 1.25 (which represents a 25% increase). The calculation works as follows: 60% × 1.25 = 75%. The team's new performance target should be a 75% completion rate, representing a meaningful improvement from their previous month's performance.\n\n***\n\nNow that we know the target completion rate for this month is 75%, we can calculate how many tasks the team should aim to complete. Assuming the team still has 120 tasks assigned for this month, we multiply: 75% × 120 = 0.75 × 120 = 90 tasks. This is the specific target the team should establish as their completion goal.\n\n***\n\nLet us verify this calculation to ensure accuracy: the team improved from 72 completed tasks last month to a goal of 90 completed tasks this month. The difference is 90 - 72 = 18 additional tasks, which represents a 25% increase in their completion rate. This verification confirms that our calculations are correct and aligned with the stated improvement goal.\n\n***\n\nTherefore, to achieve their improvement goal of 25%, the team should aim to complete 90 tasks this month. This is a concrete, measurable target that represents genuine progress from their previous month's dismal 60% completion rate. By achieving this goal of 90 completed tasks, the team will successfully demonstrate the improvement needed to save the failing product and restore confidence in their capabilities.",
    'claude_reward': True,
    'constraint': 'There should be 9 paragraphs. Paragraphs are separated with the markdown divider: ***\tThe first word of your response should be the word maximum.',
    'constraint_type': 'multi',
    'dataset': 'ifeval',
    'ground_truth': "[{'instruction_id': ['length_constraints:number_paragraphs', 'first_word:first_word_answer'], 'kwargs': [{'num_paragraphs': 9}, {'first_word': 'maximum'}]}]",
    'key': 'personas_math_easy_l3cvc3fxsysaetmszqndmdor',
    'messages': [{'content': 'The project manager of a failing product noticed that their team completed only 60% of their tasks last month. This month, the team aims to improve their performance by 25%. If the team originally had 120 tasks to complete last month, how many tasks should the team aim to complete this month to achieve their improvement goal? There should be 9 paragraphs. Paragraphs are separated with the markdown divider: *** The first word of your response should be the word maximum.',
        'role': 'user'}]
}
state = {'metrics': {}}
print(reward_fn(completion=completions, parser=parser, state=state, info=info))

2026-04-24 16:29:50.942 | DEBUG    | __main__:reward_fn:41 - Completion: <think>

</think>

**Step 1 - Identify All Constraints**:
- The response should have 9 paragraphs.
- Paragraphs are separated with the markdown divider: ***.
- The first word of the response should be the word "maximum".

**Step 2 - Check Content Correctness**:
- The content correctly calculates the number of tasks the team should aim to complete this month.
- The calculation is accurate: 60% of 120 tasks = 72 tasks completed last month.
- A 25% improvement on the completion rate (60% + 25% of 60% = 75%) results in 75% of 120 tasks = 90 tasks.
- The reasoning is sound and logically follows from the given information.

**Step 3 - Verify Each Constraint Individually**:
- **Number of paragraphs**: The draft response has 9 paragraphs, which satisfies the constraint.
- **Paragraph separator**: Each paragraph is separated by the markdown divider `***`, which satisfies the constraint.
- **First word**: The first word of 

1.0


In [11]:
# test CORRECT and claude_reward is True
info['claude_reward'] = True
print(reward_fn(completion=completions, parser=parser, state=state, info=info))
# test CORRECT and claude_reward is False
info['claude_reward'] = False
print(reward_fn(completion=completions, parser=parser, state=state, info=info))


2026-04-24 16:30:01.972 | DEBUG    | __main__:reward_fn:41 - Completion: <think>

</think>

**Step 1 - Identify All Constraints**:
- The response should have 9 paragraphs.
- Paragraphs are separated with the markdown divider: ***.
- The first word of the response should be the word "maximum".

**Step 2 - Check Content Correctness**:
- The content correctly calculates the number of tasks the team should aim to complete this month.
- The calculation is accurate: 60% of 120 tasks = 72 tasks completed last month.
- A 25% improvement on the completion rate (60% + 25% of 60% = 75%) results in 75% of 120 tasks = 90 tasks.
- The reasoning is sound and logically follows from the given information.

**Step 3 - Verify Each Constraint Individually**:
- **Number of paragraphs**: The draft response has 9 paragraphs, which satisfies the constraint.
- **Paragraph separator**: Each paragraph is separated by the markdown divider `***`, which satisfies the constraint.
- **First word**: The first word of 

1.0
0.0


In [189]:
# actual_answer is same as draft_response
actual_answer = "<|ADAPTER_RESPONSE_START|>Maximum effort is required to turn around a failing product, and understanding the numerical targets is essential for success. The project manager must calculate precisely how many tasks the team should aim to complete this month based on their previous performance and improvement goals. This mathematical problem requires careful analysis of both the historical completion rates and the projected growth targets.\n\n***\n\nLast month, the team's performance was disappointing, with only 60% task completion. Starting with 120 total tasks assigned for the month, this represents a significant shortfall that prompted the need for improvement strategies. The project manager recognized this failure rate and decided that action was necessary to get the team back on track.\n\n***\n\nTo determine the actual number of tasks completed last month, we must calculate 60% of the 120 assigned tasks. This calculation is straightforward: 0.60 multiplied by 120 equals 72 tasks. Therefore, the team successfully completed 72 tasks while failing to complete 48 tasks, leaving a notable gap in productivity.\n\n***\n\nThis month, the team has set an ambitious goal to improve their performance by 25%. This improvement target represents management's commitment to better execution and team efficiency. The question now becomes how to apply this 25% improvement to the existing completion rate of 60%.\n\n***\n\nWhen we say the team wants to improve their performance by 25%, this means they want to increase their completion rate by 25% of the current rate, not by adding 25 percentage points. In other words, the improvement is calculated as a percentage of the existing 60% completion rate. This is the standard interpretation of such improvement goals in business contexts.\n\n***\n\nTo calculate the new target completion rate, we take the previous month's completion rate of 60% and multiply it by 1.25 (which represents a 25% increase). The calculation works as follows: 60% × 1.25 = 75%. The team's new performance target should be a 75% completion rate, representing a meaningful improvement from their previous month's performance.\n\n***\n\nNow that we know the target completion rate for this month is 75%, we can calculate how many tasks the team should aim to complete. Assuming the team still has 120 tasks assigned for this month, we multiply: 75% × 120 = 0.75 × 120 = 90 tasks. This is the specific target the team should establish as their completion goal.\n\n***\n\nLet us verify this calculation to ensure accuracy: the team improved from 72 completed tasks last month to a goal of 90 completed tasks this month. The difference is 90 - 72 = 18 additional tasks, which represents a 25% increase in their completion rate. This verification confirms that our calculations are correct and aligned with the stated improvement goal.\n\n***\n\nTherefore, to achieve their improvement goal of 25%, the team should aim to complete 90 tasks this month. This is a concrete, measurable target that represents genuine progress from their previous month's dismal 60% completion rate. By achieving this goal of 90 completed tasks, the team will successfully demonstrate the improvement needed to save the failing product and restore confidence in their capabilities.<|ADAPTER_RESPONSE_END|>"
completions[0].content = actual_answer
print(reward_fn(completion=completions, parser=parser, state=state, info=info))

2026-04-22 21:14:28.525 | DEBUG    | __main__:reward_fn:41 - Completion: <|ADAPTER_RESPONSE_START|>Maximum effort is required to turn around a failing product, and understanding the numerical targets is essential for success. The project manager must calculate precisely how many tasks the team should aim to complete this month based on their previous performance and improvement goals. This mathematical problem requires careful analysis of both the historical completion rates and the projected growth targets.

***

Last month, the team's performance was disappointing, with only 60% task completion. Starting with 120 total tasks assigned for the month, this represents a significant shortfall that prompted the need for improvement strategies. The project manager recognized this failure rate and decided that action was necessary to get the team back on track.

***

To determine the actual number of tasks completed last month, we must calculate 60% of the 120 assigned tasks. This calculatio

0.0


In [190]:
# actual_answer is different from draft_response, and incorrect, and claude_reward is False. Should return 0.5
actual_answer = "<|ADAPTER_RESPONSE_START|>doesnt matter<|ADAPTER_RESPONSE_END|>"
completions[0].content = actual_answer
info['claude_reward'] = False
print(reward_fn(completion=completions, parser=parser, state=state, info=info))

2026-04-22 21:14:28.667 | DEBUG    | __main__:reward_fn:41 - Completion: <|ADAPTER_RESPONSE_START|>doesnt matter<|ADAPTER_RESPONSE_END|>
2026-04-22 21:14:28.668 | DEBUG    | __main__:reward_fn:43 - Response: doesnt matter
2026-04-22 21:14:28.668 | DEBUG    | __main__:reward_fn:45 - GT: [{'instruction_id': ['length_constraints:number_paragraphs', 'first_word:first_word_answer'], 'kwargs': [{'num_paragraphs': 9}, {'first_word': 'maximum'}]}]
2026-04-22 21:14:28.669 | DEBUG    | __main__:reward_fn:52 - claude reward: False
2026-04-22 21:14:28.669 | DEBUG    | __main__:reward_fn:68 - output_example: OutputExample(instruction_id_list=['length_constraints:number_paragraphs', 'first_word:first_word_answer'], prompt='The project manager of a failing product noticed that their team completed only 60% of their tasks last month. This month, the team aims to improve their performance by 25%. If the team originally had 120 tasks to complete last month, how many tasks should the team aim to complete

0.5


In [191]:
# actual_answer is different from draft_response, and correct, and claude_reward is False. Should return 1.0
actual_answer = f"<|ADAPTER_RESPONSE_START|>{info['claude_response']}<|ADAPTER_RESPONSE_END|>"
completions[0].content = actual_answer
info['claude_reward'] = False
info['claude_response'] = 'bad answer'
print(reward_fn(completion=completions, parser=parser, state=state, info=info))



2026-04-22 21:14:29.387 | DEBUG    | __main__:reward_fn:41 - Completion: <|ADAPTER_RESPONSE_START|>Maximum effort is required to turn around a failing product, and understanding the numerical targets is essential for success. The project manager must calculate precisely how many tasks the team should aim to complete this month based on their previous performance and improvement goals. This mathematical problem requires careful analysis of both the historical completion rates and the projected growth targets.

***

Last month, the team's performance was disappointing, with only 60% task completion. Starting with 120 total tasks assigned for the month, this represents a significant shortfall that prompted the need for improvement strategies. The project manager recognized this failure rate and decided that action was necessary to get the team back on track.

***

To determine the actual number of tasks completed last month, we must calculate 60% of the 120 assigned tasks. This calculatio

1.0


In [12]:
# need to pass parser to rubric, else it fails silently
rubric = vf.Rubric(funcs=[reward_fn, get_lgtm_count, get_fixme_count], weights=[1.0, 0.0, 0.0], parser=parser)
rubric


In [193]:
env = vf.SingleTurnEnv(
    dataset=dataset,
    parser=parser,
    system_prompt=SYSTEM_PROMPT,
    rubric=rubric,
)


Map:   0%|          | 0/5000 [00:00<?, ? examples/s]

Map:   0%|          | 0/5000 [00:00<?, ? examples/s]

In [194]:
env

In [195]:
env = vf.SingleTurnEnv(
    dataset=train_dataset,
    eval_dataset=val_dataset,
    parser=parser,
    system_prompt=SYSTEM_PROMPT,
    rubric=rubric,
)
env

Flattening the indices:   0%|          | 0/3999 [00:00<?, ? examples/s]

Map:   0%|          | 0/3999 [00:00<?, ? examples/s]

Map:   0%|          | 0/3999 [00:00<?, ? examples/s]

Flattening the indices:   0%|          | 0/1001 [00:00<?, ? examples/s]

Map:   0%|          | 0/1001 [00:00<?, ? examples/s]

Map:   0%|          | 0/1001 [00:00<?, ? examples/s]

In [196]:
env.evaluate?

Signature:
env.evaluate(
    client: 'Client | ClientConfig',
    model: 'str',
    sampling_args: 'SamplingArgs | None' = None,
    num_examples: 'int' = -1,
    rollouts_per_example: 'int' = 1,
    max_concurrent: 'int' = -1,
    results_path: 'Path | None' = None,
    state_columns: 'list[str] | None' = None,
    save_results: 'bool' = False,
    push_to_hf_hub: 'bool' = False,
    hf_hub_dataset_name: 'str | None' = None,
    independent_scoring: 'bool' = False,
    max_retries: 'int' = 0,
    on_start: 'StartCallback | None' = None,
    on_progress: 'ProgressCallback | list[ProgressCallback] | None' = None,
    on_log: 'LogCallback | None' = None,
    **kwargs,
) -> 'GenerateOutputs'
Docstring:
Evaluate model on the Environment evaluation dataset.

Args:
    on_progress: Progress callback(s). None uses the default tqdm progress bar.
        A single callback replaces the default. A list of callbacks runs
        alongside the default.
File:      /workspace/home/lab/rawhad/api-adap

In [198]:
from openai import OpenAI
client = OpenAI(base_url="http://localhost:8000/v1", api_key="")
response = client.chat.completions.create(
    model="Qwen/Qwen3-4B",
    messages=[
        {"role": "user", "content": "Hello, how are you?"},
    ],
)
response.choices[0].message.content

'<think>\nOkay, the user greeted me with "Hello, how are you?" I need to respond appropriately. First, I should acknowledge their greeting. I should say I\'m an AI assistant and mention that I\'m here to help. I should keep the tone friendly and open. Maybe add an emoji to keep it approachable. Let them know I\'m ready to assist with any questions or tasks they have. Make sure the response is concise but welcoming. Check for any grammar errors. Alright, that should cover it.\n</think>\n\nHello! I\'m an AI assistant and I\'m here to help. I\'m doing well, thank you! How can I assist you today? 😊'

In [199]:
# no think
response = client.chat.completions.create(
    model="Qwen/Qwen3-4B",
    messages=[
        {"role": "user", "content": "Hello, how are you?/no_think"},
    ],
)
response.choices[0].message.content

"<think>\n\n</think>\n\nHello! I'm just a language model, so I don't have feelings or a physical form, but I'm here and ready to help! How can I assist you today? 😊"

In [200]:
from pathlib import Path
from verifiers.types import ClientConfig

client_config = ClientConfig(
    client_type="openai_chat_completions",
    api_base_url="http://localhost:8000/v1",
    api_key="",
)

res = await env.evaluate(
    client=client_config,
    model="Qwen/Qwen3-4B",
    sampling_args=vf.SamplingArgs(max_tokens=2048),
    num_examples=10,
    rollouts_per_example=1,
    max_concurrent=10,
    results_path=Path("tmp/ifbench-verifiers-eval/qwen3-4b.jsonl"),
)
res

Processing 10 groups (10 total rollouts):   0%|                                                                  | 0/10 [00:00<?, ?it/s, reward=?]

2026-04-22 21:20:35.414 | DEBUG    | __main__:reward_fn:41 - Completion: <think>
Okay, let's tackle this step by step. The user's prompt is about Maria arranging guests from 5 countries at 3 tables, each with equal numbers from each country. Total guests are 60. The answer needs to be under 38 sentences, end with "stick", and no commas.

First, I check the content. The draft response correctly calculates 60 divided by 5 countries gives 12 per country. Then divides 12 by 3 tables, getting 4 per table. The math checks out. The explanation is clear and correct.

Now, the constraints. The response must be less than 38 sentences. Let me count. The draft has several paragraphs. Let me see: the first paragraph is 4 sentences, then the next is 4, then 5, then 4, then 3. Total is around 20 sentences, which is under 38. Good.

The last word must be "stick". The draft ends with "stick", so that's correct.

No commas allowed. The draft uses periods and other punctuation, but no commas. So that's s

{'outputs': [{'example_id': 0,
   'prompt': [SystemMessage(role='system', content='\nYou are a helpful assistant. Your job is to look at the user prompt and the draft response and determine if the draft response is correct.\n\nYou MUST think carefully inside your reasoning before outputting your final answer. Follow these evaluation steps:\n\n**Step 1 - Identify All Constraints**: Read the user prompt thoroughly and list EVERY explicit constraint, formatting requirement, and instruction. Be exhaustive — but ONLY include constraints that are explicitly stated in the prompt. Do NOT invent or infer constraints that are not present. Common constraint types include:\n- Required keywords that must appear (with specific frequencies) or must NOT appear\n- Word count, sentence count, paragraph count, section count, or bullet point count requirements\n- Structural formatting (titles wrapped in specific markers, sections with specific labels, bullet points, headers, bigram wrapping in double angu

In [ ]:
res.keys()


dict_keys(['outputs', 'metadata'])

In [57]:
sum(x['reward'] for x in res['outputs'])

0.0

In [201]:
res = env.evaluate_sync(
    client=client_config,
    model="Qwen/Qwen3-4B",
    sampling_args=vf.SamplingArgs(max_tokens=1000),
    num_examples=1,
    rollouts_per_example=1,
)

Processing 1 groups (1 total rollouts):   0%|                                                                     | 0/1 [00:00<?, ?it/s, reward=?]Aborted rollout due to ModelError() -> NotFoundError("Error code: 404 - {'error': {'message': 'The model `qwen3-8b` does not exist.', 'type': 'NotFoundError', 'param': 'model', 'code': 404}}")
2026-04-22 21:29:03.589 | ERROR    | __main__:reward_fn:71 - Error in reward_fn
Traceback (most recent call last):

  File "/home/lab/.local/share/uv/python/cpython-3.10.18-linux-x86_64-gnu/lib/python3.10/runpy.py", line 196, in _run_module_as_main
    return _run_code(code, main_globals, None,
           │         │     └ {'__name__': '__main__', '__doc__': 'Entry point for launching an IPython kernel.\n\nThis is separate from the ipykernel pack...
           │         └ <code object <module> at 0x7f75a3da9b00, file "/workspace/home/lab/rawhad/api-adapter/.venv/lib/python3.10/site-packages/ipyk...
           └ <function _run_code at 0x7f75a3d9d3f0>
  F